In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as f

In [6]:
sequence_length = 4
batch_size = 1
input_dim = 512
d_model = 512
x = torch.randn( (batch_size, sequence_length, input_dim) )
## x= positional encoder output ( that goes to our encoder not the initial input )
print(x)
print(x.shape)

tensor([[[-0.4260,  1.3356, -1.5173,  ...,  1.2309, -0.6944, -0.4787],
         [-1.5152,  1.5453,  0.6018,  ..., -1.5728,  0.1070, -1.1597],
         [ 0.2930,  0.0159, -0.6384,  ...,  1.0472, -1.7400, -1.1391],
         [ 1.1506, -0.9407, -0.7817,  ...,  0.3358,  0.9835,  0.0028]]])
torch.Size([1, 4, 512])


QKV later = q+k+v stacked on top of each other. why? Because transformers want to process all tokens in parallel instead of one-by-one like RNNs.

For a sentence like:

“I love deep learning”

each word gets turned into:

a Query (Q) → “what am I looking for?”
a Key (K) → “what info do I contain?”
a Value (V) → “what actual info should be passed?”

We stack them into matrices because matrix multiplication on GPUs is insanely fast.

so in attention = softmax (q.k^t / ....)v----> here linear algebra can go brrr using gpu

In [7]:
qkv_layer = nn.Linear(input_dim , 3 * d_model)
##simple linear neural net

In [9]:
qkv = qkv_layer(x)
qkv.shape

torch.Size([1, 4, 1536])

In [11]:
num_heads = 8
head_dim = d_model // num_heads
qkv = qkv.reshape(batch_size, sequence_length, num_heads, 3 * head_dim)
qkv.shape

torch.Size([1, 4, 8, 192])

In [12]:
qkv = qkv.permute(0, 2, 1, 3) # [batch_size, num_heads, sequence_length, 3*head_dim]
qkv.shape

torch.Size([1, 8, 4, 192])

In [13]:
#break it down into q,k,v seperate
q, k, v = qkv.chunk(3, dim=-1)
q.shape, k.shape, v.shape

(torch.Size([1, 8, 4, 64]),
 torch.Size([1, 8, 4, 64]),
 torch.Size([1, 8, 4, 64]))

Self Attention for multiple heads
self attention = softmax= (q.k^t/d_k^1/2   + Mask )------
new v = selfattention.v

In [18]:
d_k= k.shape[-1]
d_k
scaled = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(d_k)
scaled.shape

torch.Size([1, 8, 4, 4])

#Mask


In [19]:

mask = torch.full(scaled.size() , float('-inf'))
mask = torch.triu(mask, diagonal=1)
mask[0][1] # mask for input to a single head

tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])

In [20]:
scaled += mask

#Softmax
pytorch built in

In [23]:
attention = f.softmax(scaled, dim=-1)

In [24]:
attention.shape

torch.Size([1, 8, 4, 4])

In [25]:
attention

tensor([[[[1.0000, 0.0000, 0.0000, 0.0000],
          [0.3940, 0.6060, 0.0000, 0.0000],
          [0.3242, 0.2704, 0.4054, 0.0000],
          [0.3353, 0.2802, 0.2551, 0.1294]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4533, 0.5467, 0.0000, 0.0000],
          [0.3000, 0.4452, 0.2548, 0.0000],
          [0.2419, 0.1637, 0.3531, 0.2413]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5628, 0.4372, 0.0000, 0.0000],
          [0.4087, 0.1891, 0.4022, 0.0000],
          [0.3017, 0.1747, 0.2730, 0.2506]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.7049, 0.2951, 0.0000, 0.0000],
          [0.3599, 0.2649, 0.3751, 0.0000],
          [0.2458, 0.3859, 0.1660, 0.2024]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5604, 0.4396, 0.0000, 0.0000],
          [0.2865, 0.3841, 0.3294, 0.0000],
          [0.3616, 0.2206, 0.2246, 0.1932]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5855, 0.4145, 0.0000, 0.0000],
          [0.2550, 0.3

In [27]:
attention[0][0] #shape of a single head in decoder

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.3940, 0.6060, 0.0000, 0.0000],
        [0.3242, 0.2704, 0.4054, 0.0000],
        [0.3353, 0.2802, 0.2551, 0.1294]], grad_fn=<SelectBackward0>)

In [28]:
values = torch.matmul(attention, v)
values.shape

torch.Size([1, 8, 4, 64])

#Summing it up in a class

In [34]:
import torch
import torch.nn as nn
import math

def scaled_dot_product(q, k, v, mask=None):
    d_k = q.size()[-1]
    scaled = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(d_k)
    if mask is not None:
        scaled += mask
    attention = f.softmax(scaled, dim=-1)
    values = torch.matmul(attention, v)
    return values, attention

class MultiheadAttention(nn.Module):

    def __init__(self, input_dim, d_model, num_heads):
        super().__init__()
        self.input_dim = input_dim
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv_layer = nn.Linear(input_dim , 3 * d_model)
        self.linear_layer = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        batch_size, sequence_length, input_dim = x.size()
        print(f"x.size(): {x.size()}")
        qkv = self.qkv_layer(x)
        print(f"qkv.size(): {qkv.size()}")
        qkv = qkv.reshape(batch_size, sequence_length, self.num_heads, 3 * self.head_dim)
        print(f"qkv.size(): {qkv.size()}")
        qkv = qkv.permute(0, 2, 1, 3)
        print(f"qkv.size(): {qkv.size()}")
        q, k, v = qkv.chunk(3, dim=-1)
        print(f"q size: {q.size()}, k size: {k.size()}, v size: {v.size()}, ")
        values, attention = scaled_dot_product(q, k, v, mask)
        print(f"values.size(): {values.size()}, attention.size:{ attention.size()} ")
        values = values.reshape(batch_size, sequence_length, self.num_heads * self.head_dim)
        print(f"values.size(): {values.size()}")
        out = self.linear_layer(values)
        print(f"out.size(): {out.size()}")
        return out

In [35]:
input_dim = 1024
d_model = 512
num_heads = 8

batch_size = 30
sequence_length = 5
x = torch.randn( (batch_size, sequence_length, input_dim) )

model = MultiheadAttention(input_dim, d_model, num_heads)
out = model.forward(x)

x.size(): torch.Size([30, 5, 1024])
qkv.size(): torch.Size([30, 5, 1536])
qkv.size(): torch.Size([30, 5, 8, 192])
qkv.size(): torch.Size([30, 8, 5, 192])
q size: torch.Size([30, 8, 5, 64]), k size: torch.Size([30, 8, 5, 64]), v size: torch.Size([30, 8, 5, 64]), 
values.size(): torch.Size([30, 8, 5, 64]), attention.size:torch.Size([30, 8, 5, 5]) 
values.size(): torch.Size([30, 5, 512])
out.size(): torch.Size([30, 5, 512])
